> **🎯 Important**
>
**Durée estimée** : 6 à 8 heures  
**Prérequis** : Toutes les notions du Module 3 (3.1, 3.2, 3.3, 3.4)  
**Objectif** : réaliser un audit complet d'un dataset RH puis construire un pipeline de préparation ML prêt pour la production


# 🎯 Contexte

Tu es **data scientist junior** chez **TalentCorp**, une entreprise de services de 1500 salariés. La directrice des Ressources Humaines te convoque :

> « On perd trop d'employés. **Je veux comprendre qui part, pourquoi, et pouvoir prédire les départs à l'avance** pour agir avant qu'ils ne démissionnent. 
> J'ai une extraction de notre SIRH, mais elle est… disons… pas parfaite. 
> Fais-moi d'abord un **audit complet** de ces données, puis prépare-les pour qu'on puisse y brancher un modèle de ML. »

Tu disposes du fichier `employes_rh.csv` avec les données de ~1500 employés, dont une colonne `attrition` qui vaut 1 si l'employé est parti de l'entreprise.

## 📋 Ce que tu dois livrer

Deux "livrables" à la fin du TP :

1. **Un notebook d'audit** : typologie, stats descriptives, diagnostic des problèmes, observations clés
2. **Un pipeline de préparation** : code sklearn propre, prêt à être utilisé pour le ML

**Mission accomplie** = la directrice peut te donner n'importe quel nouveau fichier d'employés avec les mêmes colonnes, et tu peux produire un `X_train, X_test, y_train, y_test` avec **une seule fonction**.

# 📊 Le dataset

| Colonne | Description |
|---|---|
| `id_employe` | Identifiant unique (code alphanumérique) |
| `nom_complet` | Nom et prénom |
| `age` | Âge en années |
| `genre` | M, F, Autre |
| `departement` | Service (Ventes, R&D, RH, Marketing, Production, IT) |
| `niveau_etudes` | Bac, Licence, Master, Doctorat |
| `anciennete_annees` | Années dans l'entreprise |
| `distance_km` | Distance domicile-travail |
| `salaire_mensuel` | Salaire brut mensuel (€) |
| `heures_supp_semaine` | Heures sup par semaine |
| `satisfaction` | Très insatisfait, Insatisfait, Neutre, Satisfait, Très satisfait |
| `nb_formations` | Formations suivies (0-10) |
| `teletravail_jours` | Jours de télétravail / semaine |
| `date_embauche` | Date d'embauche |
| `matricule_interne` | Code numérique interne |
| `pays` | Pays de travail |
| `attrition` | 0 = présent, 1 = parti (**cible**) |

> **⚠️ Attention**
>
## ⚠️ Ce dataset contient (volontairement) tous les problèmes du monde réel
- Valeurs manquantes avec patterns différents
- Doublons parfaits
- Noms mal formatés (espaces, casse)
- Outliers (erreurs de saisie)
- Typos dans les modalités catégorielles
- Valeurs incohérentes (anciennetés négatives !)
- Colonnes quasi-constantes et inutiles

**Ton rôle** : tout détecter, tout traiter, et documenter ce que tu fais.


# 🛠️ Mise en route

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pandas.api.types import CategoricalDtype
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.max_columns', 20)

print("✅ Environnement prêt")

## 📥 Préparation : téléchargement des données

La cellule ci-dessous télécharge automatiquement les datasets nécessaires si ils ne sont pas déjà présents localement. Cela permet de faire marcher le notebook **aussi bien en local qu'en Google Colab**.

In [ ]:
# 📥 Téléchargement automatique des datasets (utile pour Colab)
import os, urllib.request

if not os.path.exists('ressources_tp/employes_rh.csv'):
    os.makedirs('ressources_tp', exist_ok=True)
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/Digit-Factory/python-ia-formation/main/ressources_eleves/module_03/ressources_tp/employes_rh.csv',
        'ressources_tp/employes_rh.csv'
    )
    print(f"✅ Dataset téléchargé : employes_rh.csv")
else:
    print(f"✅ Dataset déjà présent : employes_rh.csv")

In [ ]:
# Chargement silencieux pour le rendu
df = pd.read_csv("ressources_tp/employes_rh.csv")
print(f"Dataset chargé : {df.shape}")

In [ ]:
# Charger le dataset
df = pd.read_csv("ressources_tp/employes_rh.csv")
print(f"Dataset chargé : {df.shape}")

---

# Partie 1 — Audit initial (Notion 3.1)

## 🎯 Objectif

Comprendre ce qu'on a reçu. **Rien ne commence sans cet audit**.

## ✏️ Étape 1.1 — Inspection rapide

> **ℹ️ Note**
>
## 📝 À faire

1. Affiche les **dimensions** du DataFrame.
2. Affiche les **5 premières lignes**.
3. Affiche `df.info()`.
4. Affiche `df.dtypes`.

In [ ]:
# TODO: Étape 1.1

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 1.2 — Audit typologique

> **ℹ️ Note**
>
## 📝 À faire

Crée un **tableau d'audit** en markdown (ou un DataFrame) qui, pour chaque colonne du dataset, indique :

- Le **type statistique** (quanti continue/discrète, quali nominale/ordinale, textuelle, temporelle, identifiant)
- Le **dtype actuel** dans Pandas
- L'**action recommandée** (convertir en category, en datetime, en ordinale, supprimer, etc.)

In [ ]:
# TODO: Étape 1.2 — audit typologique

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 1.3 — Diagnostic complet

> **ℹ️ Note**
>
## 📝 À faire

Reprends ta fonction `diagnostiquer()` de la Notion 3.3 (ou recrée-la) et applique-la au dataset. Commente ensuite :

- Combien de **doublons** ?
- Quelles colonnes ont des **NaN** et combien ?
- Y a-t-il des **colonnes constantes** ?
- La **répartition de la cible** `attrition` (classes équilibrées ou non ?)

In [ ]:
# TODO: Étape 1.3 — diagnostic

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

---

# Partie 2 — Statistiques descriptives (Notion 3.2)

## ✏️ Étape 2.1 — Analyse univariée

> **ℹ️ Note**
>
## 📝 À faire

Pour 3 variables quantitatives de ton choix (par exemple `salaire_mensuel`, `age`, `anciennete_annees`), produis :

1. Les **statistiques** : moyenne, médiane, écart-type, skewness.
2. Un **histogramme** pour chacune.

Que remarques-tu ? Y a-t-il des distributions asymétriques, des outliers visibles ?

In [ ]:
# TODO: Étape 2.1

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 2.2 — Analyse bivariée cible

> **ℹ️ Note**
>
## 📝 À faire

C'est la partie **la plus intéressante** pour la RH : quelles variables semblent liées à l'attrition ?

1. **Quanti vs cible** : pour chaque variable numérique intéressante (`heures_supp_semaine`, `distance_km`, `anciennete_annees`, `salaire_mensuel`), compare les moyennes entre `attrition=0` et `attrition=1`. Les partants ont-ils des profils différents ?
   
   *Indice : `df.groupby("attrition")[col].mean()`*

2. **Quali vs cible** : pour `departement` et `niveau_etudes`, calcule le taux d'attrition par groupe.

In [ ]:
# TODO: Étape 2.2

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 2.3 — Matrice de corrélation

> **ℹ️ Note**
>
## 📝 À faire

Calcule et visualise la matrice de corrélation des variables quantitatives **y compris la cible `attrition`**. Quelles variables sont les plus corrélées avec `attrition` ?

In [ ]:
# TODO: Étape 2.3

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

---

# Partie 3 — Nettoyage (Notion 3.3)

## ✏️ Étape 3.1 — Supprimer les colonnes inutiles

> **ℹ️ Note**
>
## 📝 À faire

Pour préparer le ML, **supprime les colonnes inutiles** :

- `id_employe` et `nom_complet` : identifiants/textes, non utiles pour prédire
- `matricule_interne` : code numérique sans signification
- `pays` : colonne constante

**Attention** : tu peux **garder `id_employe` comme index** si tu veux tracer les prédictions ensuite.

In [ ]:
# TODO: Étape 3.1

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 3.2 — Supprimer les doublons

> **ℹ️ Note**
>
## 📝 À faire

Supprime les doublons et affiche combien ont été retirés.

In [ ]:
# TODO: Étape 3.2

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 3.3 — Normaliser les textes catégoriels

> **ℹ️ Note**
>
## 📝 À faire

La colonne `satisfaction` contient des valeurs mal normalisées : `"Satisfait"`, `"satisfait"`, `"SATISFAIT"` sont la même valeur.

1. Normalise `satisfaction` pour que tout soit sur la même forme (propose `.str.strip().str.title()` pour remettre en "Titre").
2. Vérifie les valeurs uniques après normalisation.

In [ ]:
# TODO: Étape 3.3

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 3.4 — Traiter les incohérences

> **ℹ️ Note**
>
## 📝 À faire

1. Identifie les valeurs **négatives** dans `anciennete_annees` (c'est impossible).
2. Remplace-les par **0** (valeur par défaut cohérente : nouvelle embauche).

In [ ]:
# TODO: Étape 3.4

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 3.5 — Traiter les outliers

> **ℹ️ Note**
>
## 📝 À faire

1. Sur `salaire_mensuel` : détecte les outliers avec la méthode **IQR**. Combien ? Passe-les à `NaN` (on va les imputer juste après).
2. Sur `distance_km` : applique un **capping** à 100 km (valeur maximale plausible).

In [ ]:
# TODO: Étape 3.5

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 3.6 — Imputer les valeurs manquantes

> **ℹ️ Note**
>
## 📝 À faire

Applique la **bonne stratégie** pour chaque colonne avec NaN :

1. `salaire_mensuel` : imputation par la **médiane par `niveau_etudes`** (le salaire dépend du niveau → MAR).
2. `distance_km` : imputation par la **médiane globale** (MCAR).
3. `satisfaction` : imputation par le **mode**.

In [ ]:
# TODO: Étape 3.6

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

---

# Partie 4 — Pipeline de préparation ML (Notion 3.4)

## ✏️ Étape 4.1 — Conversion des types

> **ℹ️ Note**
>
## 📝 À faire

Avant le pipeline :

1. Convertis `date_embauche` en **datetime**.
2. Convertis les catégorielles nominales (`genre`, `departement`) en `category`.
3. Convertis les **ordinales** en `CategoricalDtype` ordonné :
   - `niveau_etudes` : Bac < Licence < Master < Doctorat
   - `satisfaction` : Très Insatisfait < Insatisfait < Neutre < Satisfait < Très Satisfait

In [ ]:
# TODO: Étape 4.1

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 4.2 — Extraction de features à partir de la date

> **ℹ️ Note**
>
## 📝 À faire

À partir de `date_embauche`, crée :

1. `mois_embauche` (1-12)
2. `trimestre_embauche` (1-4)
3. Encode `mois_embauche` de manière **cyclique** en `mois_sin` et `mois_cos`.

Ensuite, **supprime** la colonne `date_embauche` (on a déjà `anciennete_annees` qui capture la même info en plus utile).

In [ ]:
# TODO: Étape 4.2

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 4.3 — Construire le `ColumnTransformer`

> **ℹ️ Note**
>
## 📝 À faire

Construis un `ColumnTransformer` qui applique :

- `StandardScaler` aux colonnes numériques continues (`age`, `anciennete_annees`, `distance_km`, `salaire_mensuel`, `heures_supp_semaine`)
- `OneHotEncoder` aux nominales (`genre`, `departement`)
- `OrdinalEncoder` aux ordinales (`niveau_etudes`, `satisfaction`) avec l'ordre explicite
- `passthrough` pour les features déjà numériques (`nb_formations`, `teletravail_jours`, `mois_sin`, `mois_cos`)

Tu dois gérer le cas où une nouvelle modalité apparaîtrait en test avec `handle_unknown="ignore"`.

In [ ]:
# TODO: Étape 4.3

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 4.4 — Split + Application

> **ℹ️ Note**
>
## 📝 À faire

1. Sépare `X` (toutes les features) et `y` (`attrition`).
2. Fais un `train_test_split` **stratifié** (80/20, random_state=42, stratify=y) — la stratification est cruciale parce que la cible est déséquilibrée.
3. Vérifie que les proportions de `attrition=1` sont identiques dans train et test.
4. Applique `fit_transform` au preprocessor sur `X_train`, puis `transform` sur `X_test`.
5. Affiche la shape finale et les noms des features générées.

In [ ]:
# TODO: Étape 4.4

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 4.5 — Encapsuler dans une fonction

> **ℹ️ Note**
>
## 📝 À faire

Pour finir proprement, encapsule **tout le pipeline de nettoyage + préparation** dans une fonction `preparer_donnees_rh(df)` qui :

1. Prend un DataFrame brut en entrée (comme le CSV original)
2. Fait tout le traitement : nettoyage, conversion, feature engineering
3. Retourne un DataFrame prêt à être passé au `ColumnTransformer`

Cette fonction doit être **réutilisable** sur n'importe quel nouveau batch de données du même format.

In [ ]:
# TODO: Étape 4.5 — fonction réutilisable

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

---

# Partie 5 — Livrable final : note méthodologique

> **ℹ️ Note**
>
## 📝 À faire

Rédige une **note méthodologique** d'une page (400-600 mots) pour la directrice RH, avec :

1. **Contexte** : la mission, les données
2. **Diagnostic** : les problèmes que tu as identifiés
3. **Traitements appliqués** : ce que tu as fait, dans quel ordre, et pourquoi
4. **Features finales** : la liste des variables retenues pour le modèle
5. **Observations clés** : les patterns identifiés en analyse descriptive
6. **Limites** et recommandations

C'est exactement le livrable attendu en entreprise.


```markdown
# Note méthodologique — Préparation des données RH

## 1. Contexte
[À rédiger]

## 2. Diagnostic initial
[À rédiger]

## 3. Traitements appliqués
[À rédiger]

## 4. Features finales
[À rédiger]

## 5. Observations clés
[À rédiger]

## 6. Limites et recommandations
[À rédiger]
```

> **💡 Astuce**
>
## ✅ Exemple de note méthodologique

```markdown
# Note méthodologique — Préparation des données RH pour prédiction d'attrition

## 1. Contexte

Dans le cadre de la mission de réduction de l'attrition, j'ai procédé à l'audit 
et la préparation du fichier SIRH (~1500 employés, 17 colonnes). L'objectif est 
de préparer les données pour un modèle prédictif, avec un pipeline reproductible 
pouvant tourner chaque mois sur les nouvelles extractions.

## 2. Diagnostic initial

L'audit a révélé plusieurs problèmes :
- **5 doublons parfaits** (même employé plusieurs fois dans l'export)
- **124 valeurs manquantes** au total, réparties sur 3 colonnes :
  - `salaire_mensuel` : 49 NaN, corrélés à l'âge (plus de données manquantes 
    chez les < 30 ans — pattern MAR)
  - `distance_km` : 30 NaN aléatoires
  - `satisfaction` : 45 NaN
- **5 outliers de salaire** (erreurs de saisie type 10x le salaire réel)
- **2 outliers de distance** (> 200 km, aberrant)
- **3 valeurs négatives d'ancienneté** (impossibles, probablement erreurs)
- **Typographie hétérogène** dans satisfaction ("Satisfait", "satisfait", "SATISFAIT")
- **3 colonnes inutiles** : `matricule_interne` (code sans sens), `pays` (constant), 
  `nom_complet` (texte non exploitable)

## 3. Traitements appliqués

Dans l'ordre :
1. Suppression des colonnes inutiles
2. Suppression des doublons
3. Normalisation de `satisfaction` (casse uniformisée)
4. Correction des ancienntés négatives (mises à 0)
5. Outliers salaire : détectés par IQR, mis à NaN
6. Outliers distance : cappés à 100 km
7. Imputation conditionnelle : salaire par médiane de `niveau_etudes` (MAR), 
   distance par médiane globale (MCAR), satisfaction par mode
8. Conversion des types : dates, catégories ordonnées (niveau_etudes, satisfaction)
9. Feature engineering : mois d'embauche encodé cycliquement (sin/cos)

## 4. Features finales

Le dataset final contient ~18 features après encoding :
- 5 numériques standardisées (age, ancienneté, distance, salaire, heures sup)
- ~8 colonnes one-hot (genre × 3, département × 6)
- 2 ordinales encodées (niveau d'études, satisfaction)
- 4 passthrough (nb_formations, télétravail, mois_sin, mois_cos)

## 5. Observations clés

- **Taux d'attrition : ~11%** (dataset déséquilibré, stratification au split 
  indispensable)
- Les partants font **significativement plus d'heures supplémentaires** 
  (indicateur de burn-out potentiel)
- Les partants ont **moins d'ancienneté** (effet "nouveau qui part")
- Les partants habitent **plus loin** du bureau
- Pas de lien simple entre niveau d'études et attrition

## 6. Limites et recommandations

**Limites** :
- Pas d'information sur les **évaluations de performance**
- Pas de données sur les **primes** ou augmentations récentes
- Taille modeste (1500 lignes) — augmentera la variance du modèle

**Recommandations** :
- Enrichir le dataset avec les évaluations annuelles
- Segmenter l'analyse par département (patterns probablement différents)
- Mettre en place un **monitoring** des features (dérive possible au fil du temps)

Le pipeline `preparer_donnees_rh()` est prêt à être utilisé en production. 
Il peut consommer n'importe quel export du même format et produire des données 
propres directement consommables par le modèle ML.
```


---

# 🎓 Bilan du TP

## 🏆 Ce que tu viens d'accomplir

Tu as réalisé ton **premier audit de données complet** :

- ✅ **Diagnostic typologique** de 17 colonnes hétérogènes
- ✅ **Analyse descriptive** univariée, bivariée, multivariée
- ✅ **Nettoyage méthodique** : colonnes inutiles, doublons, normalisation, outliers, NaN
- ✅ **Pipeline scikit-learn** fonctionnel et réutilisable
- ✅ **Fonction d'ingestion** `preparer_donnees_rh()` pour la production
- ✅ **Note méthodologique** professionnelle

## 🔍 Les notions mobilisées

| Notion | Utilisée dans |
|---|---|
| **3.1 Typologie** | Audit, choix des encodings, suppression colonnes inutiles |
| **3.2 Stats descriptives** | Univariée, bivariée cible, matrice de corrélation |
| **3.3 Nettoyage** | Doublons, NaN (MCAR/MAR), outliers, normalisation textes |
| **3.4 Préparation ML** | Scaling, encoding, features de date, split stratifié, pipeline |

## 🚀 La suite — Module 4 : Machine Learning supervisé

Tu as maintenant **tout ce qu'il faut** pour passer à la modélisation. Au Module 4, on va :

- Transformer cette préparation en **modèle prédictif** réel
- Apprendre **classification vs régression**
- Comparer plusieurs algorithmes (régression logistique, arbres, Random Forest, XGBoost)
- Maîtriser les **métriques d'évaluation** (accuracy, precision/recall, ROC-AUC)
- Optimiser les **hyperparamètres** (grid search, cross-validation)

Et tout ça s'**empilera directement** sur le pipeline que tu viens de créer !

> **💡 Astuce**
>
## 💡 Défi pour consolider

Tes acquis sur ce TP te permettent déjà de **faire un modèle réel** :

```python
from sklearn.linear_model import LogisticRegression

modele = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

modele.fit(X_train, y_train)
print(f"Accuracy sur test : {modele.score(X_test, y_test):.3f}")
```

Essaie ! Tu devrais obtenir autour de 85-90% d'accuracy. On verra au Module 4 
comment c'est une métrique trompeuse sur un dataset déséquilibré, et comment 
faire mieux.


**🎉 Bravo d'être arrivé·e au bout — tu as vraiment le niveau data scientist junior maintenant.**